In [53]:
import importlib
import user_class

importlib.reload(user_class)
from user_class import User

print(User.__module__)
print(User.__init__.__code__.co_varnames)


user_class
('self', 'user_id', 'bankroll', 'bet_history', 'pitch_index', 'table')


In [54]:
import os, time
import boto3
from datetime import datetime, timezone
from decimal import Decimal
from botocore.exceptions import ClientError
from fastapi import HTTPException

In [55]:
DEFAULT_BANKROLL = float(os.getenv("DEFAULT_BANKROLL", "100"))

# create function to return DynamoDB table storing user session information
def get_table():
    dynamodb = boto3.resource("dynamodb", region_name=os.getenv("AWS_REGION", "us-east-1"))
    table = dynamodb.Table(os.getenv("SESSIONS_TABLE", "PitchBettingSessions"))
    return table

In [56]:
def create_user(user_id, table):
    # create a new user with the default bankroll and empty bet history
    user = User(
        user_id=user_id,
        bankroll=DEFAULT_BANKROLL,
        bet_history=[],
        pitch_index=0,
        table=table
    )

    user.save_to_db()

    return user

In [57]:
print(User.__module__)
print(User.__init__.__code__.co_varnames)


user_class
('self', 'user_id', 'bankroll', 'bet_history', 'pitch_index', 'table')


In [58]:
def get_user(user_id):
    table = get_table()
    resp = table.get_item(Key={"user_id": user_id})
    if "Item" in resp:
        item = resp["Item"]

        # create a User object from the DynamoDB item
        user = User(
            user_id=user_id,
            bankroll=float(item["bankroll"]),
            bet_history=item["bet_history"],
            pitch_index=int(item["pitch_index"]),
            table=table
        )

    # otherwise, create a new user with the default bankroll and empty bet history
    else:
        user = create_user(user_id, table)

    return user

user = get_user("topher-test2")

In [59]:
user.advance_pitch()

In [ ]:
user.get_pitch_index()